In [ ]:
#--- Late Fusion --- 
# The trend and residuals components both use the late fusion strategy

#--- Import necessary packages ---
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL 
import numpy as np
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Model
from tensorflow.keras.layers import InputLayer, LSTM, Dense, Dropout, Input, Concatenate, Bidirectional                                                                                                                                                                 
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import IPython.display as display
import os            
import random
import tensorflow as tf
import copy
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow.keras.backend as K
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Customized loss function ---
def customTrendLoss(lambda_, delta, penalty_factor, threshold):
    def loss(y_true, y_pred):
        # The formula of Basic Error
        error = y_true - y_pred
        abs_error = K.abs(error)

        # The formula of Huber Loss
        is_small = K.less_equal(abs_error, delta)
        huber = tf.where(is_small, 0.5 * K.square(error), delta * (abs_error - 0.5 * delta))
        huber_loss = K.mean(huber)

        # The formula of Directional Penalty
        def thresholdedSign(x):
            return tf.where(x < -threshold, -1.0, tf.where(x > threshold, 1.0, 0.0))

        directionTrue = thresholdedSign(y_true)
        directionPred = thresholdedSign(y_pred)
        directionMismatch = K.cast(K.not_equal(directionTrue, directionPred), K.floatx())
        direction_penalty = K.mean(directionMismatch * penalty_factor * abs_error)

        # The formula of combined custom components
        custom = huber_loss + direction_penalty

        # The formula of MSE for stability
        mseLoss = K.mean(K.square(error))

        finalLoss = lambda_ * mseLoss + (1.0 - lambda_) * custom
        return finalLoss
    
    return loss
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Makes the model's output stable ---

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Defines all significant variables ---

# Defines the timesteps variable
timesteps = 8
# Defines the interval variable
interval = 7
# Defines the output Timesteps variable
outputTimesteps = 8
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Defines all function for the evaluation metrics (MAPE, SMAPE, and trend accuracy) ---

# Defines function for the MAPE metric formula (%)
def MAPE(y_true, y_prediction):
    epsilon = 1e-10  
    return np.mean(np.abs((y_true - y_prediction) / (y_true + epsilon))) * 100

# Defines function for the SMAPE metric formula (%)
def SMAPE(y_true, y_pred):
    epsilon = 1e-10  
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + epsilon))

# Defines function for the trend accuracy
def trendAccuracy(y_actualBefore, y_actual, y_predictedBefore, y_predicted, threshold):
  
    trendActual = ((y_actual-y_actualBefore)/y_actualBefore) * 100
    trendPredicted = ((y_predicted-y_predictedBefore)/y_predictedBefore) * 100

    def labelTrend(trendValues, threshold):
        labels = np.zeros_like(trendValues, dtype=int)
        labels[trendValues > threshold] = 1 
        labels[trendValues < -threshold] = -1
        return labels
    
    actual_labels = labelTrend(trendActual, threshold)
    predicted_labels = labelTrend(trendPredicted, threshold)

    correct_predictions = np.sum(actual_labels == predicted_labels)
    accuracy = correct_predictions / len(actual_labels)

    return accuracy
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Preprocesses the target or y variable (Daily WTI Crude Oil Spot Price) ---

# Loads the Daily WTI Crude Oil Spot Price from a .csv file (from 2015 until 2019) (a period of 5 years) 
priceData = pd.read_csv('Crude Oil WTI Spot Price (5 Years) - copy.csv') 
# print(priceData.shape) # Shape [1254, 2]

# Transforms the variable in the 'Date' column into executable variable (for filling the missing values in the next step)
priceData['Date'] = pd.to_datetime(priceData['Date'], format='%d-%b-%y')
# Transforms the 'Date' column into an index of the dataframe
priceData.set_index('Date', inplace=True)
# print(priceData.shape) # Shape [1254, 1]

# Fills and handles the missing values (missing values including holidays in weekdays and weekends)
allDates = pd.date_range(start=priceData.index.min(), end=priceData.index.max(), freq='D') # freq = 'D' means all avilable days between the start and end date
priceData = priceData.reindex(allDates)
priceData.index.name = 'Date'
# Fills missing values using interpolation method
priceData['Price'] = priceData['Price'].interpolate(method='linear').round(2) 
# print(priceData.shape) # Shape [1825, 1] # The day of the date starts from Monday and ends on Friday (1825 days + 2 days = 1827 / 7 = 261 weeks in total)
# Removes the last 5 rows because of the incompleteness (no saturday and sunday on that week) and exceeding the year of 2019
priceData = priceData.iloc[:-5, :]
# print(priceData.shape) # Shape [1820, 1]

# Defines a function to visualize the daily crude oil spot prices (original and decomposition components)
# This function is for visualization purpose only
def visualizePriceDecomposition(priceData):
    plt.figure(figsize=(12, 8))
    # All components (original daily crude oil spot price) (additive methodology)
    plt.subplot(4, 1, 1)
    plt.plot(priceData['Price'], label='WTI Crude Oil Spot Price')
    plt.title('Original Prices', fontsize=10)
    plt.legend()
    # Trend component
    plt.subplot(4, 1, 2)
    plt.plot(priceData['trend'], label='Trend')
    plt.title('Trend Component', fontsize=10)
    plt.legend()
    # Seasonal component
    plt.subplot(4, 1, 3)
    plt.plot(priceData['seasonal'], label='Seasonality')
    plt.title('Seasonal Component', fontsize=10)
    plt.legend()
    # Residuals component
    plt.subplot(4, 1, 4)
    plt.plot(priceData['residual'], label='Residuals')
    plt.title('Residual Component', fontsize=10)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Calls and implements the decomposition package (decomposing into trend, seasonal, and residual components) and rounds these components into two decimals place after
stl = STL(priceData['Price'], period=28) # 28 represents the number of days in a month 
result = stl.fit()
priceData['trend'] = result.trend.round(2)
priceData['seasonal'] = result.seasonal.round(2)
priceData['residual'] = result.resid.round(2)
# print(priceData.shape) # Shape [1820, 4]
# Calls the visualization function
# visualizePriceDecomposition(priceData)

# Adds the 'price_diff', 'trend_diff', 'seasonal_diff', 'residual_diff' columns because the research wants to predict the diff rather than the price
# The _diff uses a weekly shift (7 days shifting)
# Eventhough the dataset is daily price, this research wants to make it to have a weekly behavior
priceData['price_diff'] = priceData['Price'] - priceData['Price'].shift(7)
priceData['trend_diff'] = priceData['trend'] - priceData['trend'].shift(7)
priceData['seasonal_diff'] = priceData['seasonal'] - priceData['seasonal'].shift(7)
priceData['residual_diff'] = priceData['residual'] - priceData['residual'].shift(7)
# print(priceData.shape) # Shape [1820, 8]
# Removes the first 7 rows because of the NaN values on the '_diff' columns
priceData = priceData.iloc[7:]
# print(priceData.shape) # Shape [1813, 8]

# Note: the variables below will not be used on the fusion strategies
# Adds a column with the day of week status (Monday, Tuesday, etc.)
priceData['DayOfWeek'] = priceData.index.day_name()
daysOrder = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
priceData = pd.get_dummies(priceData, columns=['DayOfWeek'])
# Renames columns to remove the 'DayOfWeek_' prefix
renameMapping = {f'DayOfWeek_{day}': day for day in daysOrder}
priceData.rename(columns=renameMapping, inplace=True)
# Converts boolean 'True/False' values to integer '1/0' (one-hot enconding) for simplicity
priceData[daysOrder] = priceData[daysOrder].astype(int)
# Reorders the columns order on the dataframe 
priceData = priceData[['Price', 'trend', 'seasonal', 'residual', 'price_diff', 'trend_diff', 'seasonal_diff', 'residual_diff'] + daysOrder]
# print(priceData.shape) # Shape [1813, 15]
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Preprocess the x variables (macroeconomics features) ---

# Loads a csv file that contains the list of the choosen macroeconomics variables from granger casuality test
selectedMacroVariables = pd.read_csv('CausingVariables.csv', header=None) 
# Converts the dataframe into a list, removes the first variable on the list (the header of the list) and adds a 'DATE' column into the list
selectedMacroVariables = selectedMacroVariables[0].tolist()
selectedMacroVariables = selectedMacroVariables[1:]
selectedMacroVariables.insert(0, 'DATE')
# print(len(selectedMacroVariables)) # Shape [431]

# Loads a csv file containing the macroeconomics variables dataset (differencing once) (already stationary and non-constant)
macroeconomicsData = pd.read_csv("MacroeconomicsDailyFilteredStationaryCleaned.csv") 
# print(macroeconomicsData.shape) # Shape [1813, 8867]

# Chooses the macroeconomics variables that only contained in the list (filtering the macroeconomics dataset)
finalMacroeconomicData = macroeconomicsData.copy()
finalMacroeconomicData = finalMacroeconomicData[selectedMacroVariables]
# print(finalMacroeconomicData.shape) # Shape [1813, 431]
# Transforms the 'DATE' from columns to index of the dataframe
finalMacroeconomicData.set_index("DATE", inplace=True) 
# print(finalMacroeconomicData.shape) # Shape [1813, 430]
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Preprocess the x variables (textual features) (FinBERT Embeddings) (can be original or GPT-generated features) ---

# Loads a .csv file containing weekly sentence embedding within the called feautures
sentenceEmbeddingsData = pd.read_csv('Original Headline Sentence Embedding FinBERTku.csv')
# print(sentenceEmbeddingsData.shape) # Shape [1826, 770] # 5 years between 2015 and 2019 contains 1826 days

# Removes the first 11 rows and last 2 rows aligning with the price dataset
sentenceEmbeddingsData = sentenceEmbeddingsData.iloc[11: , : ]
sentenceEmbeddingsData = sentenceEmbeddingsData.iloc[ :-2 , : ]
# print(sentenceEmbeddingsData.shape) # Shape [1813, 770]

# Ensures the 'Date' column is in the correct and executable format
# sentenceEmbeddingsData['Date'] = pd.to_datetime(sentenceEmbeddingsData['Date'])
# Transforms the 'Date' from column to index of the dataframe 
sentenceEmbeddingsData.set_index('Date', inplace=True)
# print(sentenceEmbeddingsData.shape) # Shape [1813, 769]
# Deletes the first column of the dataframe (the raw textual feature)
sentenceEmbeddingsData=sentenceEmbeddingsData.iloc[ : , 1: ] 
# print(sentenceEmbeddingsData.shape) # Shape [1813, 768]
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Preprocesses all the X variables and y variable so it can be putted on the RNN models (train, validation, and test) phases ---

# Defines the Standard Scaler independently for each y variable (the one hot encoding variables are not scalled)
priceScaler = StandardScaler()
trendScaler = StandardScaler()
seasonalScaler = StandardScaler()
residualScaler = StandardScaler()
price_diffScaler = StandardScaler()
trend_diffScaler = StandardScaler()
seasonal_diffScaler = StandardScaler()
residual_diffScaler = StandardScaler()
# Defines the Standard Scaler for the X variables (one scaller for all the macroeconomics variables)
macroeconomicScaler = StandardScaler()

# Scales each column with defined variables (independently)
priceData.loc[:, 'Price'] = priceScaler.fit_transform(priceData[['Price']])
priceData.loc[:, 'trend'] = trendScaler.fit_transform(priceData[['trend']])
priceData.loc[:, 'seasonal'] = seasonalScaler.fit_transform(priceData[['seasonal']])
priceData.loc[:, 'residual'] = residualScaler.fit_transform(priceData[['residual']])
priceData.loc[:, 'price_diff'] = price_diffScaler.fit_transform(priceData[['price_diff']])
priceData.loc[:, 'trend_diff'] = trend_diffScaler.fit_transform(priceData[['trend_diff']])
priceData.loc[:, 'seasonal_diff'] = seasonal_diffScaler.fit_transform(priceData[['seasonal_diff']])
priceData.loc[:, 'residual_diff'] = residual_diffScaler.fit_transform(priceData[['residual_diff']])
# Scales all columns with a defined variable
finalMacroeconomicData[:] = macroeconomicScaler.fit_transform(finalMacroeconomicData)

# Defines the sequence functions (trend, seasonal, residual) with an interval value (7 days interval)
# Uses previous 8 weeks sequence to predict the next 8 weeks sequence
# Defines the trend sequence function
def createTrendSequence(pricedf, macroeconomicsdf, sentenceEmbeddingsdf, timesteps, outputTimesteps, interval, targetColumn, priceColumn):
    X, y, price, priceBefore = [], [], [], []
    choosenPriceColumns = [priceColumn, targetColumn]
    
    for i in range(len(pricedf) - (timesteps+outputTimesteps) * interval):
        Xprice = pricedf[choosenPriceColumns].iloc[i:i + timesteps * interval: interval].values
        Xmacroeconomics = macroeconomicsdf.iloc[i:i + timesteps * interval: interval].values
        XsentenceEmbeddings = sentenceEmbeddingsdf.iloc[i:i + timesteps * interval: interval].values
    
        X.append(np.hstack((Xprice, Xmacroeconomics, XsentenceEmbeddings)))
       
        y.append(pricedf[[targetColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        price.append(pricedf[[priceColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        priceBefore.append(pricedf[[priceColumn]].iloc[i+timesteps*interval-interval].values.flatten())

    return np.array(X), np.array(y), np.array(price), np.array(priceBefore)

# Defines the seasonal and residual sequence functions (price as the target variable)
def createPriceSequence(pricedf, timesteps, outputTimesteps, interval, targetColumn, priceColumn):
    X, y, price, priceBefore = [], [], [], []
    choosenPriceColumns = [priceColumn, targetColumn]
    
    for i in range(len(pricedf) - (timesteps+outputTimesteps) * interval):
        Xprice = pricedf[choosenPriceColumns].iloc[i:i + timesteps * interval: interval].values
        
        X.append(Xprice)
        
        y.append(pricedf[[priceColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        price.append(pricedf[[priceColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        priceBefore.append(pricedf[[priceColumn]].iloc[i+timesteps*interval-interval].values.flatten())

    return np.array(X), np.array(y), np.array(price), np.array(priceBefore)

# Defines the seasonal and residual sequence functions (price as the target variable)
def createPriceResidualsSequence(pricedf, sentenceEmbeddingsdf, timesteps, outputTimesteps, interval, targetColumn, priceColumn):
    X, y, price, priceBefore = [], [], [], []
    choosenPriceColumns = [priceColumn, targetColumn]
    
    for i in range(len(pricedf) - (timesteps+outputTimesteps) * interval):
        Xprice = pricedf[choosenPriceColumns].iloc[i:i + timesteps * interval: interval].values
        XsentenceEmbeddings = sentenceEmbeddingsdf.iloc[i:i + timesteps * interval: interval].values
        
        X.append(np.hstack((Xprice, XsentenceEmbeddings)))
        
        y.append(pricedf[[priceColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        price.append(pricedf[[priceColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        priceBefore.append(pricedf[[priceColumn]].iloc[i+timesteps*interval-interval].values.flatten())

    return np.array(X), np.array(y), np.array(price), np.array(priceBefore)

# Calls the function and creates sequences for the train, validation, and test datasets
X_trend, y_trend, y_priceTrend, y_priceBeforeTrend = createTrendSequence(priceData, finalMacroeconomicData, sentenceEmbeddingsData, timesteps, outputTimesteps, interval, 'trend_diff', 'trend')
X_seasonal, y_seasonal, y_priceSeasonal, y_priceBeforeSeasonal = createPriceSequence(priceData, timesteps, outputTimesteps, interval, 'seasonal_diff', 'seasonal')
X_residual, y_residual, y_priceResidual, y_priceBeforeResidual = createPriceResidualsSequence(priceData, sentenceEmbeddingsData, timesteps, outputTimesteps, interval, 'residual_diff', 'residual')

# # Prints the shape of each X sequence and y sequence
# print(f"X_trend shape: {X_trend.shape}") # Shape [1701, 8, 1200]
# print(f"y_trend shape: {y_trend.shape}") # Shape [1701, 8] 
# print(f"y_priceTrend shape: {y_priceTrend.shape}") # Shape [1701, 8]
# print(f"y_priceBeforeTrend shape: {y_priceBeforeTrend.shape}") # Shape [1701, 1]
# print(f"X_seasonal shape: {X_seasonal.shape}") # Shape [1701, 8, 2]
# print(f"y_seasonal shape: {y_seasonal.shape}") # Shape [1701, 8]
# print(f"y_priceSeasonal shape: {y_priceSeasonal.shape}") # Shape [1701, 8] 
# print(f"y_priceBeforeSeasonal shape: {y_priceBeforeSeasonal.shape}") # Shape [1701, 1]
# print(f"X_residual shape: {X_residual.shape}") # Shape [1701, 8, 770]
# print(f"y_residual shape: {y_residual.shape}") # Shape [1701, 8]
# print(f"y_priceResidual shape: {y_priceResidual.shape}") # Shape [1701, 8]
# print(f"y_priceBeforeResidual shape: {y_priceBeforeResidual.shape}") # Shape [1701, 1]

# Defines lists for train, validation, and test datasets for each component (3 folds cross-validation)
X_trendTrains, y_trendTrains, y_priceTrendTrains, X_trendValidations, y_trendValidations, y_priceTrendValidations, X_trendTests, y_trendTests, y_priceTrendTests = [], [], [], [], [], [], [], [], []
X_seasonalTrains, y_seasonalTrains, y_priceSeasonalTrains, X_seasonalValidations, y_seasonalValidations, y_priceSeasonalValidations, X_seasonalTests, y_seasonalTests, y_priceSeasonalTests = [], [], [], [], [], [], [], [], []
X_residualTrains, y_residualTrains, y_priceResidualTrains, X_residualValidations, y_residualValidations, y_priceResidualValidations, X_residualTests, y_residualTests, y_priceResidualTests = [], [], [], [], [], [], [], [], []

# Defines lists for train, validation, test datasets for each component (price before)
y_priceTrendBeforeTrains, y_priceTrendBeforeValidations, y_priceTrendBeforeTests = [], [], []
y_priceSeasonalBeforeTrains, y_priceSeasonalBeforeValidations, y_priceSeasonalBeforeTests = [], [], []
y_priceResidualBeforeTrains, y_priceResidualBeforeValidations, y_priceResidualBeforeTests = [], [], []

# Defines the fold boundaries (3 folds cross-validations)
folds = [
    {"train": (0, 676), "validation": (676, 932), "test": (932, 1188)},
    {"train": (0, 932), "validation": (932, 1188), "test": (1188, 1444)},
    {"train": (0, 1188), "validation": (1188, 1444), "test": (1444, 1700)}
] 

# Defines a function to split data into three folds for cross-validations for train, validation, and test datasets
def splitFolds(X, y, y_price, y_priceBefore, folds):
    X_trains, y_trains, y_priceTrains, y_priceBeforeTrains = [], [], [], []
    X_validations, y_validations, y_priceValidations, y_priceBeforeValidations = [], [], [], []
    X_tests, y_tests, y_priceTests, y_priceBeforeTests = [], [], [], []

    for fold in folds:
        trainStart, trainEnd = fold["train"]
        valStart, valEnd = fold["validation"]
        testStart, testEnd = fold["test"]
        # Train
        X_trains.append(X[trainStart:trainEnd])
        y_trains.append(y[trainStart:trainEnd])
        y_priceTrains.append(y_price[trainStart:trainEnd])
        y_priceBeforeTrains.append(y_priceBefore[trainStart:trainEnd])
        # Validation
        X_validations.append(X[valStart:valEnd])
        y_validations.append(y[valStart:valEnd])
        y_priceValidations.append(y_price[valStart:valEnd])
        y_priceBeforeValidations.append(y_priceBefore[valStart:valEnd])
        # Test
        X_tests.append(X[testStart:testEnd])
        y_tests.append(y[testStart:testEnd])
        y_priceTests.append(y_price[testStart:testEnd])
        y_priceBeforeTests.append(y_priceBefore[testStart:testEnd])

    return (X_trains, y_trains, y_priceTrains,
            X_validations, y_validations, y_priceValidations,
            X_tests, y_tests, y_priceTests,
            y_priceBeforeTrains, y_priceBeforeValidations, y_priceBeforeTests)

# Calls the function for splitting the train, validation, and tests datasets into 3 folds
(X_trendTrains, y_trendTrains, y_priceTrendTrains,
 X_trendValidations, y_trendValidations, y_priceTrendValidations,
 X_trendTests, y_trendTests, y_priceTrendTests,
 y_priceTrendBeforeTrains, y_priceTrendBeforeValidations, y_priceTrendBeforeTests) = splitFolds(X_trend, y_trend, y_priceTrend, y_priceBeforeTrend, folds)
(X_seasonalTrains, y_seasonalTrains, y_priceSeasonalTrains,
 X_seasonalValidations, y_seasonalValidations, y_priceSeasonalValidations,
 X_seasonalTests, y_seasonalTests, y_priceSeasonalTests,
 y_priceSeasonalBeforeTrains, y_priceSeasonalBeforeValidations, y_priceSeasonalBeforeTests) = splitFolds(X_seasonal, y_seasonal, y_priceSeasonal, y_priceBeforeSeasonal, folds)
(X_residualTrains, y_residualTrains, y_priceResidualTrains,
 X_residualValidations, y_residualValidations, y_priceResidualValidations,
 X_residualTests, y_residualTests, y_priceResidualTests,
 y_priceResidualBeforeTrains, y_priceResidualBeforeValidations, y_priceResidualBeforeTests) = splitFolds(X_residual, y_residual, y_priceResidual, y_priceBeforeResidual, folds)

# Define a helper function to print fold shapes
def printFoldShapes(name, X_trains, y_trains, y_priceTrains, 
                      X_validations, y_validations, y_priceValidations, 
                      X_tests, y_tests, y_priceTests, 
                      y_priceBeforeTrains, y_priceBeforeValidations, y_priceBeforeTests):
    
    print(f"\n{name.upper()} COMPONENT")
    for i in range(len(X_trains)):
        print(f"Fold {i+1}:")
        print(f"  Train:       X={X_trains[i].shape}, y={y_trains[i].shape}, y_price={y_priceTrains[i].shape}, y_price_before={y_priceBeforeTrains[i].shape}")
        print(f"  Validation:  X={X_validations[i].shape}, y={y_validations[i].shape}, y_price={y_priceValidations[i].shape}, y_price_before={y_priceBeforeValidations[i].shape}")
        print(f"  Test:        X={X_tests[i].shape}, y={y_tests[i].shape}, y_price={y_priceTests[i].shape}, y_price_before={y_priceBeforeTests[i].shape}")

# Calls the function to print shapes for all 3 components
# printFoldShapes("trend", X_trendTrains, y_trendTrains, y_priceTrendTrains, 
#                   X_trendValidations, y_trendValidations, y_priceTrendValidations,
#                   X_trendTests, y_trendTests, y_priceTrendTests,
#                   y_priceTrendBeforeTrains, y_priceTrendBeforeValidations, y_priceTrendBeforeTests)

# printFoldShapes("seasonal", X_seasonalTrains, y_seasonalTrains, y_priceSeasonalTrains, 
#                   X_seasonalValidations, y_seasonalValidations, y_priceSeasonalValidations,
#                   X_seasonalTests, y_seasonalTests, y_priceSeasonalTests,
#                   y_priceSeasonalBeforeTrains, y_priceSeasonalBeforeValidations, y_priceSeasonalBeforeTests)

# printFoldShapes("residual", X_residualTrains, y_residualTrains, y_priceResidualTrains, 
#                   X_residualValidations, y_residualValidations, y_priceResidualValidations,
#                   X_residualTests, y_residualTests, y_priceResidualTests,
#                   y_priceResidualBeforeTrains, y_priceResidualBeforeValidations, y_priceResidualBeforeTests)

# Trend component
# Fold 01
# Shape [884, 8, 1200] [884, 8] [884, 8] [884, 1] [136, 8, 1200] [136, 8] [136, 8] [136, 1] [136, 8, 1200] [136, 8] [136, 8] [136, 1]
# Fold 02
# Shape [1156, 8, 1200] [1156, 8] [1156, 8] [1156, 1] [136, 8, 1200] [136, 8] [136, 8] [136, 1] [136, 8, 1200] [136, 8] [136, 8] [136, 1]
# Fold 03
# Shape [1428, 8, 1200] [1428, 8] [1428, 8] [1428, 1] [136, 8, 1200] [136, 8] [136, 8] [136, 1] [136, 8, 1200] [136, 8] [136, 8] [136, 1]

# Seasonal component
# Fold 01
# Shape [884, 8, 2] [884, 8] [884, 8] [884, 1] [136, 8, 2] [136, 8] [136, 8] [136, 1] [136, 8, 2] [136, 8] [136, 8] [136, 1]
# Fold 02
# Shape [1156, 8, 2] [1156, 8] [1156, 8] [1156, 1] [136, 8, 2] [136, 8] [136, 8] [136, 1] [136, 8, 2] [136, 8] [136, 8] [136, 1]
# Fold 03
# Shape [1428, 8, 2] [1428, 8] [1428, 8] [1428, 1] [136, 8, 2] [136, 8] [136, 8] [136, 1] [136, 8, 2] [136, 8] [136, 8] [136, 1]

# Residual component
# Fold 01
# Shape [884, 8, 770] [884, 8] [884, 8] [884, 1] [136, 8, 770] [136, 8] [136, 8] [136, 1] [136, 8, 770] [136, 8] [136, 8] [136, 1]
# Fold 02
# Shape [1156, 8, 770] [1156, 8] [1156, 8] [1156, 1] [136, 8, 770] [136, 8] [136, 8] [136, 1] [136, 8, 770] [136, 8] [136, 8] [136, 1]
# Fold 03
# Shape [1428, 8, 770] [1428, 8] [1428, 8] [1428, 1] [136, 8, 770] [136, 8] [136, 8] [136, 1] [136, 8, 770] [136, 8] [136, 8] [136, 1]

# Consolidates train, validation, and test datasets into dictionaries
X_trains = {'trend': X_trendTrains, 'seasonal': X_seasonalTrains, 'residual': X_residualTrains}
y_trains = {'trend': y_trendTrains, 'seasonal': y_seasonalTrains, 'residual': y_residualTrains}
y_priceTrains = {'trend': y_priceTrendTrains, 'seasonal': y_priceSeasonalTrains, 'residual': y_priceResidualTrains}
y_priceBeforeTrains = {'trend': y_priceTrendBeforeTrains, 'seasonal': y_priceSeasonalBeforeTrains, 'residual': y_priceResidualBeforeTrains}
X_validations = {'trend': X_trendValidations, 'seasonal': X_seasonalValidations, 'residual': X_residualValidations}
y_validations = {'trend': y_trendValidations, 'seasonal': y_seasonalValidations, 'residual': y_residualValidations}
y_priceValidations = {'trend': y_priceTrendValidations, 'seasonal': y_priceSeasonalValidations, 'residual': y_priceResidualValidations}
y_priceBeforeValidations = {'trend': y_priceTrendBeforeValidations, 'seasonal': y_priceSeasonalBeforeValidations, 'residual': y_priceResidualBeforeValidations}
X_tests = {'trend': X_trendTests, 'seasonal': X_seasonalTests, 'residual': X_residualTests}
y_tests = {'trend': y_trendTests, 'seasonal': y_seasonalTests, 'residual': y_residualTests}
y_priceTests = {'trend': y_priceTrendTests, 'seasonal': y_priceSeasonalTests, 'residual': y_priceResidualTests}
y_priceBeforeTests = {'trend': y_priceTrendBeforeTests, 'seasonal': y_priceSeasonalBeforeTests, 'residual': y_priceResidualBeforeTests}
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Builds the RNN Models for the trend, seasonal, and residuals independently ---
# Defines a model outputing the number of outpuTimesteps on the final layer silmutaneously

def TrendModel(timesteps, numericalFeatures, textualFeatures, outputTimesteps):
    numericalInput = Input(shape=(timesteps, numericalFeatures), name='numericalInput')
    x_numerical = LSTM(512, return_sequences=False)(numericalInput)
    x_numerical = Dropout(0.1)(x_numerical)
    y_pred_numerical = Dense(outputTimesteps, activation='linear', name='numericalPrediction')(x_numerical)

    textualInput = Input(shape=(timesteps, textualFeatures), name='textualInput')
    x_textual = LSTM(32, return_sequences=False)(textualInput)
    x_textual = Dropout(0.1)(x_textual)
    y_pred_textual = Dense(outputTimesteps, activation='linear', name='textualPrediction')(x_textual)

    fusion = Concatenate(name="fusionLayer")([y_pred_numerical, y_pred_textual])  
    fusion = Dropout(0.1)(fusion)                        
    final_output = Dense(outputTimesteps, activation='linear', name='finalOutput')(fusion)

    model = Model(inputs=[numericalInput, textualInput], outputs=final_output)
    model.compile(optimizer=Adam(),loss=customTrendLoss(lambda_=0.7, delta=1, penalty_factor=0.5, threshold=0.05))
    
    return model
    
def SeasonalModel(timesteps, features, outputTimesteps):
    model = Sequential()
    model.add(InputLayer(shape=(timesteps, features)))
    model.add(LSTM(200, return_sequences=False))
    model.add(Dense(outputTimesteps, activation='linear'))  
    model.compile(optimizer=Adam(), loss='MSE')
    return model
 
def ResidualModel(timesteps, numericalFeatures, textualFeatures, outputTimesteps):
    # Numerical input sequence (price)
    numericalInput = Input(shape=(timesteps, numericalFeatures), name='numericalInput')
    x_numerical = Bidirectional(LSTM(200, return_sequences=False))(numericalInput)
    x_numerical = Dropout(0.3)(x_numerical)

    # Textual input (last timestep embedding only)
    textualInput = Input(shape=(textualFeatures,), name='textualInput')
    x_textual = Dense(50, activation='relu')(textualInput)
    x_textual = Dropout(0.3)(x_textual)

    # Fusion and output
    merged = Concatenate(name="concatenatedFeatures")([x_numerical, x_textual])
    output = Dense(outputTimesteps, activation='linear')(merged)

    model = Model(inputs=[numericalInput, textualInput], outputs=output)
    #optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=Adam(), loss=customTrendLoss(lambda_=0.2, delta=1.5, penalty_factor=0, threshold=0))
    return model

# Defines lists to store models, training losses, and validation losses for each components-folds combination
models = {'trend': [], 'seasonal': [], 'residual': []}
trainLosses = {'trend': [], 'seasonal': [], 'residual': []}
valLosses = {'trend': [], 'seasonal': [], 'residual': []}
# Defines a directory to save the best models for each components-folds combination
modelSaveDirectory = './BestLateFusion(LSTM)/'
os.makedirs(modelSaveDirectory, exist_ok=True)

class LiveLossPlotCallback(tf.keras.callbacks.Callback):
    def __init__(self):
        self.train_losses = []
        self.val_losses = []

    def on_epoch_end(self, epoch, logs=None):
        self.train_losses.append(logs.get('loss'))
        self.val_losses.append(logs.get('val_loss'))
        plt.clf()
        plt.plot(self.train_losses, label='Training Loss')
        plt.plot(self.val_losses, label='Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss (MSE)')
        plt.title('Live Training vs Validation Loss')
        plt.legend()
        plt.grid(True)
        display.clear_output(wait=True)
        display.display(plt.gcf())

# Defines a function to train and save the best models for each components-folds combination
def trainSaveBestModel(X_train, y_train, X_validation, y_validation, priceIdx, numericalIdx, textualIdx, epochs, batch_size, featureType, idx):

    if featureType == 'trend':
        XNumerical_train = X_train.copy()
        XTextual_train = X_train.copy()
        XNumerical_validation = X_validation.copy()
        XTextual_validation = X_validation.copy()
       
        XNumerical_train = XNumerical_train[:, :, :numericalIdx]
        XTextual_train = XTextual_train[:, :, numericalIdx:numericalIdx+textualIdx]
        XNumerical_validation = XNumerical_validation[:, :, :numericalIdx]
        XTextual_validation = XTextual_validation[:, :, numericalIdx:numericalIdx+textualIdx]
    
    if featureType == 'residual':
        XNumerical_train = X_train.copy()
        XTextual_train = X_train.copy()
        XNumerical_validation = X_validation.copy()
        XTextual_validation = X_validation.copy()
       
        XNumerical_train = XNumerical_train[:, :, :numericalIdx]
        XTextual_train = XTextual_train[:, 7, numericalIdx:numericalIdx+textualIdx]
        XNumerical_validation = XNumerical_validation[:, :, :numericalIdx]
        XTextual_validation = XTextual_validation[:, 7, numericalIdx:numericalIdx+textualIdx]

    if featureType == 'trend':
        regressor = TrendModel(timesteps, numericalIdx, textualIdx, outputTimesteps)
    elif featureType == 'seasonal':
        regressor = SeasonalModel(timesteps, X_train.shape[-1], outputTimesteps)
    elif featureType == 'residual':
        regressor = ResidualModel(timesteps, numericalIdx, textualIdx, outputTimesteps)

    earlyStopping = EarlyStopping(
        monitor='val_loss', 
        patience=30,
        restore_best_weights=True,
        verbose=1
    )

    if featureType == 'trend':
        history = regressor.fit(
            [XNumerical_train, XTextual_train], y_train,
            validation_data=([XNumerical_validation, XTextual_validation], y_validation),
            epochs=epochs,
            batch_size=batch_size,
            shuffle=False,
            callbacks=[earlyStopping],
            verbose=1
        )

    elif featureType == 'residual':
        history = regressor.fit(
            [XNumerical_train, XTextual_train], y_train,
            validation_data=([XNumerical_validation, XTextual_validation], y_validation),
            epochs=epochs,
            batch_size=batch_size,
            shuffle=False,
            callbacks=[earlyStopping],
            verbose=1
        )
    else:
        history = regressor.fit(
            X_train, y_train,
            validation_data=(X_validation, y_validation),
            epochs=epochs,
            batch_size=batch_size,
            shuffle=False,
            callbacks=[earlyStopping],
            verbose=1
        )

    bestValLoss = min(history.history['val_loss'])
    print(f"Best Validation Loss for {featureType.capitalize()} Fold {idx + 1}: {bestValLoss:.3f}")
    modelSavePath = os.path.join(modelSaveDirectory, f'bestModel_{featureType}_fold0{idx + 1}.weights.h5')
    regressor.save_weights(modelSavePath)
    print(f"Saved best model weights to {modelSavePath}")

    return regressor, history.history

# Calls the train and save the best models functions for each components-folds combination
for featureType in ['trend', 'seasonal', 'residual']:
    for idx, (X_train, y_train, X_val, y_val) in enumerate(zip(X_trains[featureType], y_trains[featureType], X_validations[featureType], y_validations[featureType])):
        print(f"\nTraining and saving the best model for {featureType.capitalize()} fold {idx + 1}...")
        if featureType == 'trend':
            model, history = trainSaveBestModel(X_train, y_train, X_val, y_val, priceIdx=None, numericalIdx=432, textualIdx=768, epochs=1000, batch_size=64, featureType=featureType, idx=idx)
        elif featureType == 'seasonal':
            model, history = trainSaveBestModel(X_train, y_train, X_val, y_val, priceIdx=None, numericalIdx=None, textualIdx=None, epochs=1, batch_size=64, featureType=featureType, idx=idx)
        elif featureType == 'residual':
            model, history = trainSaveBestModel(X_train, y_train, X_val, y_val, priceIdx=None, numericalIdx=2, textualIdx=768, epochs=1000, batch_size=32, featureType=featureType, idx=idx)
        models[featureType].append(model)
        trainLosses[featureType].append(history['loss'])
        valLosses[featureType].append(history['val_loss'])

# Visualizes the train and validation losses separately for each component
for featureType in ['trend', 'seasonal', 'residual']:
    plt.figure(figsize=(10, 5))
    for idx, (trainLoss, valLoss) in enumerate(zip(trainLosses[featureType], valLosses[featureType])):
        plt.plot(range(len(trainLoss)), trainLoss, label=f'Train - Fold {idx + 1}')
        plt.plot(range(len(valLoss)), valLoss, linestyle='dashed', label=f'Validation - Fold {idx + 1}')
    plt.title(f'{featureType.capitalize()} Component: Training and Validation Loss per Epoch', fontsize=12)
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Loss (MSE)', fontsize=10)
    plt.legend(prop={'size': 8})
    plt.grid(True)
    plt.tight_layout()
    plt.show()
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Defines function to check the prediction results by outputting the csv files ---

# Defines functions to save the actual and the predicted target variable values for each folds-components combination
# There will be 72 files outputed from this function
def savePredictionsToCSV(featureType, fold, currentStep, y_priceBeforeTrain_inverse, y_trainPrediction_inverse, y_priceTrainPrediction_inverse, y_train_inverse, y_priceTrain_inverse, label):
    dfPrediction = pd.DataFrame({
        'Price Before': np.round(y_priceBeforeTrain_inverse.flatten(), 2),
        'Predicted Price Difference': np.round(y_trainPrediction_inverse.flatten(), 2),
        'Predicted Price': np.round(y_priceTrainPrediction_inverse.flatten(), 2),
        'Actual Price Difference': np.round(y_train_inverse.flatten(), 2),
        'Actual Price': np.round(y_priceTrain_inverse.flatten(), 2),
        
    })
    csvOutputPath = f'./{label}_fold0{fold + 1}_{featureType}(0{currentStep + 1}).csv'
    dfPrediction.to_csv(csvOutputPath, index=False)

# Defines functions to save the combined actual and the predicted target variable values for each folds combination
# There will be 24 files outputed from this function
def saveOverallPredictionsToCSV(fold, currentStep, y_combinedPriceTrain_inverse, y_combinedPriceTrainPrediction_inverse, label):
    # Creates a folder to save the results 
    outputDir = './BestLateFusionPredictionResults(LSTM)'
    os.makedirs(outputDir, exist_ok=True)
    dfCombined = pd.DataFrame({
        'Price': np.round(y_combinedPriceTrain_inverse.flatten(), 2),
        'Predicted Price': np.round(y_combinedPriceTrainPrediction_inverse.flatten(), 2),
    })
    csvOutputPath = os.path.join(outputDir, f'{label}_fold0{int(fold[-2:])}_overall(0{int(currentStep) + 1}).csv')
    dfCombined.to_csv(csvOutputPath, index=False)
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Defines a function to test the test datasets (each components-folds combination)  ---

def TestModel(models, X_tests, y_tests, y_priceTests, y_priceBeforeTests, trendScaler, trend_diffScaler, seasonalScaler, seasonal_diffScaler, residualScaler, residual_diffScaler, outputTimesteps):
    
    # Puts the defined scalers on dictionaries according to each feature type (trend, seasonal, and residual)
    scalers = {
        'trend': (trendScaler, trend_diffScaler),
        'seasonal': (seasonalScaler, seasonal_diffScaler),
        'residual': (residualScaler, residual_diffScaler)
    }

    # Initializes metrics dictionaries for each timesteps-folds-metrics-components combination
    initialMetrics = {
        'mae': {'fold01': [], 'fold02': [], 'fold03': []}, 
        'mse': {'fold01': [], 'fold02': [], 'fold03': []}, 
        'rmse': {'fold01': [], 'fold02': [], 'fold03': []}, 
        'mape': {'fold01': [], 'fold02': [], 'fold03': []},
        'smape': {'fold01': [], 'fold02': [], 'fold03': []}
    }
    testMetrics = {
        'trend': copy.deepcopy(initialMetrics),
        'seasonal': copy.deepcopy(initialMetrics),
        'residual': copy.deepcopy(initialMetrics)
    }
    priceTestMetrics = copy.deepcopy(testMetrics)

    # Initializes metrics dictionaries for each timesteps-metrics-folds (additive methodology) (trend + seasonal + residuals prices)
    combinedPriceTestMetrics = {'mae': {}, 'mse': {}, 'rmse': {}, 'mape': {}, 'smape': {}, 'trendAcc01': {}, 'trendAcc02': {}, 'trendAcc03': {}}

    # Initializes dictionaries for each fold combination (additive methodology) (trend + seasonal + residuals prices)
    y_combinedPriceTests_inverse = {'fold01': {}, 'fold02': {}, 'fold03': {}}
    y_combinedPriceTestPredictions_inverse = {'fold01': {}, 'fold02': {}, 'fold03': {}}
    y_combinedPriceBeforeTests_inverse = {'fold01': {}, 'fold02': {}, 'fold03': {}}

    # Loops through all the components
    for featureType in ['trend', 'seasonal', 'residual']:
        priceScaler, price_diffScaler = scalers[featureType]
        print(f"\nTesting on {featureType.capitalize()} Models...")

        # Loops through all the folds
        for idx, (X_test, y_test, y_priceTest, y_priceBeforeTest, model) in enumerate(zip(
            X_tests[featureType], y_tests[featureType], y_priceTests[featureType], y_priceBeforeTests[featureType], models[featureType])):
        
            foldKey = f'fold0{idx + 1}'
            print(f"\nTesting on Fold 0{idx + 1}...")

            # Predicts all future timesteps at once by calling the trained model
            if featureType == 'trend':
                y_testPrediction = model.predict([X_test[:, :, :432], X_test[:, :, 432:1200]])
            elif featureType == 'residual':
                y_testPrediction = model.predict([X_test[:, :, :2], X_test[:, 7, 2:770]])
            else:
                y_testPrediction = model.predict(X_test)

            # Because in the trend component this program predicts the price difference
            if featureType in ['trend']:
                # Inverses transform predictions and actual prices difference
                y_testPrediction_inverse = np.round(price_diffScaler.inverse_transform(y_testPrediction), 2)
                y_test_inverse = np.round(price_diffScaler.inverse_transform(y_test), 2)
                # Restores and inverses transform actual price before values
                y_priceBeforeTest_inverse = priceScaler.inverse_transform(y_priceBeforeTest.reshape(-1, 1))
                # Computes cumulative sum across timesteps starting from retrieves price before values
                y_priceTestPrediction_inverse = np.cumsum(np.hstack([y_priceBeforeTest_inverse, y_testPrediction_inverse]), axis=1)[:, 1:]
                y_priceTest_inverse = np.round(priceScaler.inverse_transform(y_priceTest), 2) 

            # Because in the seasonal and residual components this program directly predicts the price
            elif featureType in ['seasonal', 'residual']:
                # Note: the y_train_inverse and the y_priceTrain_inverse will have the same values
                # Inverses transform predictions and actual prices
                y_testPrediction_inverse = np.round(priceScaler.inverse_transform(y_testPrediction), 2)
                y_test_inverse = np.round(priceScaler.inverse_transform(y_test), 2)
                # Restores and inverses transform actual price before values
                y_priceBeforeTest_inverse = priceScaler.inverse_transform(y_priceBeforeTest.reshape(-1, 1))
                # Because we predicts the price, we just put the prediction into the price prediction variable
                y_priceTestPrediction_inverse = np.round(priceScaler.inverse_transform(y_testPrediction), 2)
                y_priceTest_inverse = np.round(priceScaler.inverse_transform(y_priceTest), 2)
                if featureType in ['seasonal']:
                    y_priceTestPrediction_inverse = np.round(priceScaler.inverse_transform(y_priceTest), 2)
        
            # Computes and stores the metrices for predicted price differences and predicted prices
            for t in range(outputTimesteps):
                testMetrics[featureType]['mae'][foldKey].append(mean_absolute_error(y_test_inverse[:, t], y_testPrediction_inverse[:, t]))
                testMetrics[featureType]['mse'][foldKey].append(mean_squared_error(y_test_inverse[:, t], y_testPrediction_inverse[:, t]))
                testMetrics[featureType]['rmse'][foldKey].append(np.sqrt(mean_squared_error(y_test_inverse[:, t], y_testPrediction_inverse[:, t])))
                testMetrics[featureType]['mape'][foldKey].append(MAPE(y_test_inverse[:, t], y_testPrediction_inverse[:, t]))
                testMetrics[featureType]['smape'][foldKey].append(SMAPE(y_test_inverse[:, t], y_testPrediction_inverse[:, t]))

                priceTestMetrics[featureType]['mae'][foldKey].append(mean_absolute_error(y_priceTest_inverse[:, t], y_priceTestPrediction_inverse[:, t]))
                priceTestMetrics[featureType]['mse'][foldKey].append(mean_squared_error(y_priceTest_inverse[:, t], y_priceTestPrediction_inverse[:, t]))
                priceTestMetrics[featureType]['rmse'][foldKey].append(np.sqrt(mean_squared_error(y_priceTest_inverse[:, t], y_priceTestPrediction_inverse[:, t])))
                priceTestMetrics[featureType]['mape'][foldKey].append(MAPE(y_priceTest_inverse[:, t], y_priceTestPrediction_inverse[:, t]))  
                priceTestMetrics[featureType]['smape'][foldKey].append(SMAPE(y_priceTest_inverse[:, t], y_priceTestPrediction_inverse[:, t]))   

            # # Saves the prediction to a .CSV file for all timesteps
            # # Should be outputed 72 .csv files
            # for t in range(outputTimesteps):
            #     savePredictionsToCSV(
            #         featureType, idx, t, 
            #         y_priceBeforeTest_inverse,
            #         y_testPrediction_inverse[:, t].reshape(-1, 1), 
            #         y_priceTestPrediction_inverse[:, t].reshape(-1, 1), 
            #         y_test_inverse[:, t].reshape(-1, 1), 
            #         y_priceTest_inverse[:, t].reshape(-1, 1), 
            #         'test'
            #     )

            # Combines trend + seasonal + residual components (additive methodology)
            for t in range(outputTimesteps):
                if t not in y_combinedPriceTests_inverse[foldKey]:
                    y_combinedPriceTests_inverse[foldKey][t] = y_priceTest_inverse[:,t]
                    y_combinedPriceTestPredictions_inverse[foldKey][t] = y_priceTestPrediction_inverse[:,t]
                else:
                    y_combinedPriceTests_inverse[foldKey][t] += y_priceTest_inverse[:,t]
                    y_combinedPriceTestPredictions_inverse[foldKey][t] += y_priceTestPrediction_inverse[:,t]

                # Gets the price before at t = 0 for counting the trend accuracy
                if t == 0:
                    if t not in y_combinedPriceBeforeTests_inverse[foldKey]:
                        y_combinedPriceBeforeTests_inverse[foldKey][0] = y_priceBeforeTest_inverse.flatten()
                    else:
                        y_combinedPriceBeforeTests_inverse[foldKey][0] += y_priceBeforeTest_inverse.flatten()

        # # Iterates over timesteps to print the metrics of the components-folds combination
        # for fold_num in range(len(folds)):
        #     foldKey = f'fold0{fold_num + 1}'
        #     for t in range(outputTimesteps):
        #         print(f"\n(Test) Price Metrics for {featureType.capitalize()} {foldKey.capitalize()}, Timestep {t + 1}: "
        #                 f"MAE: {priceTestMetrics[featureType]['mae'][foldKey][t]:.3f}, "
        #                 f"MSE: {priceTestMetrics[featureType]['mse'][foldKey][t]:.3f}, "
        #                 f"RMSE: {priceTestMetrics[featureType]['rmse'][foldKey][t]:.3f}, "
        #                 f"MAPE: {priceTestMetrics[featureType]['mape'][foldKey][t]:.3f}, "
        #                 f"SMAPE: {priceTestMetrics[featureType]['smape'][foldKey][t]:.3f}")
        #     print('\n') 
            
    # Saves and output a .csv file for the overall actual and predicted prices for each fold combination (additive methodology)
    # Should be outputed 24 .csv files
    for foldKey in ['fold01', 'fold02', 'fold03']:
        for t in y_combinedPriceTests_inverse[foldKey].keys():
            saveOverallPredictionsToCSV(foldKey, t, y_combinedPriceTests_inverse[foldKey][t], y_combinedPriceTestPredictions_inverse[foldKey][t], 'test')

    # Calculates the metrics of the overall components for each timesteps-folds combination
    # Loops over folds
    for foldKey in ['fold01', 'fold02', 'fold03']:
        # Loops over timesteps
        for t in range(outputTimesteps):
            if t in y_combinedPriceTests_inverse[foldKey]:
                combinedPriceTestMetricsMAE = mean_absolute_error(y_combinedPriceTests_inverse[foldKey][t], y_combinedPriceTestPredictions_inverse[foldKey][t])
                combinedPriceTestMetricsMSE = mean_squared_error(y_combinedPriceTests_inverse[foldKey][t], y_combinedPriceTestPredictions_inverse[foldKey][t])
                combinedPriceTestMetricsRMSE = np.sqrt(combinedPriceTestMetricsMSE)
                combinedPriceTestMetricsMAPE = MAPE(y_combinedPriceTests_inverse[foldKey][t], y_combinedPriceTestPredictions_inverse[foldKey][t])
                combinedPriceTestMetricsSMAPE = SMAPE(y_combinedPriceTests_inverse[foldKey][t], y_combinedPriceTestPredictions_inverse[foldKey][t])
                combinedPriceTestMetricsTrendAcc01 = trendAccuracy(
                    y_actualBefore=y_combinedPriceBeforeTests_inverse[foldKey][0], 
                    y_actual=y_combinedPriceTests_inverse[foldKey][t], 
                    y_predictedBefore=y_combinedPriceBeforeTests_inverse[foldKey][0],  
                    y_predicted=y_combinedPriceTestPredictions_inverse[foldKey][t], threshold=1)
                combinedPriceTestMetricsTrendAcc02 = trendAccuracy(
                    y_actualBefore=y_combinedPriceBeforeTests_inverse[foldKey][0], 
                    y_actual=y_combinedPriceTests_inverse[foldKey][t], 
                    y_predictedBefore=y_combinedPriceBeforeTests_inverse[foldKey][0],  
                    y_predicted=y_combinedPriceTestPredictions_inverse[foldKey][t], threshold=3)
                combinedPriceTestMetricsTrendAcc03 = trendAccuracy(
                    y_actualBefore=y_combinedPriceBeforeTests_inverse[foldKey][0], 
                    y_actual=y_combinedPriceTests_inverse[foldKey][t], 
                    y_predictedBefore=y_combinedPriceBeforeTests_inverse[foldKey][0],  
                    y_predicted=y_combinedPriceTestPredictions_inverse[foldKey][t], threshold=8.75)

                for metric, value in zip(['mae', 'mse', 'rmse', 'mape', 'smape', 'trendAcc01', 'trendAcc02', 'trendAcc03'],
                    [combinedPriceTestMetricsMAE, combinedPriceTestMetricsMSE, combinedPriceTestMetricsRMSE, combinedPriceTestMetricsMAPE, combinedPriceTestMetricsSMAPE, combinedPriceTestMetricsTrendAcc01,
                        combinedPriceTestMetricsTrendAcc02,combinedPriceTestMetricsTrendAcc03]):
                    if t not in combinedPriceTestMetrics[metric]:
                        combinedPriceTestMetrics[metric][t] = []
                    combinedPriceTestMetrics[metric][t].append(value)

            # Outputs metrics for each timesteps-folds combination
            print(f"\n===== {foldKey.upper()} | Timestep {t+1} Results =====")
            print(f"MAE     : {combinedPriceTestMetricsMAE:.4f}")
            print(f"MSE     : {combinedPriceTestMetricsMSE:.4f}")
            print(f"RMSE    : {combinedPriceTestMetricsRMSE:.4f}")
            print(f"MAPE    : {combinedPriceTestMetricsMAPE:.4f}")
            print(f"SMAPE   : {combinedPriceTestMetricsSMAPE:.4f}")
            print(f"TrendAcc @ ±1%   : {combinedPriceTestMetricsTrendAcc01:.4f}")
            print(f"TrendAcc @ ±3%   : {combinedPriceTestMetricsTrendAcc02:.4f}")
            print(f"TrendAcc @ ±8.75%: {combinedPriceTestMetricsTrendAcc03:.4f}")
            print("="*45)

    # Calculates and outputs Mean and Standard Deviation of each Metric for each timesteps (Test Phase)
    print(f"\n=== Overall Price Metrics Summary (TEST) ===")
    for t in range(outputTimesteps):
        print(f"=== Timestep {t+1} ===")
        for metric in ['mae', 'mse', 'rmse', 'mape', 'smape', 'trendAcc01', 'trendAcc02', 'trendAcc03']:
            values = combinedPriceTestMetrics[metric].get(t, [])
            if values:
                mean_val = np.mean(values)
                std_val = np.std(values)
                print(f"{metric.upper():<10} - Mean: {mean_val:.4f}, Std: {std_val:.4f}")
        print("\n")

    # Generate and save final metric summary inside the function
    mean_table = {"Timestep": list(range(1, outputTimesteps + 1))}
    std_table = {"Timestep": list(range(1, outputTimesteps + 1))}
    metric_list = ['mae', 'mse', 'rmse', 'mape', 'smape', 'trendAcc01', 'trendAcc02', 'trendAcc03']

    for metric in metric_list:
        mean_vals = []
        std_vals = []
        for t in range(outputTimesteps):
            values = combinedPriceTestMetrics[metric].get(t, [])
            mean_vals.append(np.round(np.mean(values), 4))
            std_vals.append(np.round(np.std(values), 4))
        
        mean_table[metric.upper()] = mean_vals
        std_table[metric.upper()] = std_vals

    df_mean = pd.DataFrame(mean_table)
    df_std = pd.DataFrame(std_table)

    empty_row = pd.DataFrame([[""] + [""] * (len(df_mean.columns) - 1)], columns=df_mean.columns)
    final_df = pd.concat([df_mean, empty_row, df_std], ignore_index=True)
    final_df.to_csv("./BestLateFusion(LSTM).csv", index=False)
  
    # Stacks all timestep arrays horizontally (axis=1) to form new shape for all folds
    fold01_pred = np.column_stack([y_combinedPriceTestPredictions_inverse['fold01'][t] for t in range(outputTimesteps)])
    fold01_actual = np.column_stack([y_combinedPriceTests_inverse['fold01'][t] for t in range(outputTimesteps)])
    fold01_price_before = y_combinedPriceBeforeTests_inverse['fold01'][0]

    fold02_pred = np.column_stack([y_combinedPriceTestPredictions_inverse['fold02'][t] for t in range(outputTimesteps)])
    fold02_actual = np.column_stack([y_combinedPriceTests_inverse['fold02'][t] for t in range(outputTimesteps)])
    fold02_price_before = y_combinedPriceBeforeTests_inverse['fold02'][0]

    fold03_pred = np.column_stack([y_combinedPriceTestPredictions_inverse['fold03'][t] for t in range(outputTimesteps)])
    fold03_actual = np.column_stack([y_combinedPriceTests_inverse['fold03'][t] for t in range(outputTimesteps)])
    fold03_price_before = y_combinedPriceBeforeTests_inverse['fold03'][0]

    return (fold01_pred, fold01_actual, fold01_price_before,
            fold02_pred, fold02_actual, fold02_price_before,
            fold03_pred, fold03_actual, fold03_price_before)

# Calls the function to test datasets
(fold1_pred, fold1_actual, fold1_price_before,
 fold2_pred, fold2_actual, fold2_price_before,
 fold3_pred, fold3_actual, fold3_price_before) = TestModel(
    models=models,
    X_tests=X_tests,
    y_tests=y_tests,
    y_priceTests=y_priceTests,
    y_priceBeforeTests=y_priceBeforeTests,
    trendScaler=trendScaler,
    trend_diffScaler=trend_diffScaler,
    seasonalScaler=seasonalScaler,
    seasonal_diffScaler=seasonal_diffScaler,
    residualScaler=residualScaler,
    residual_diffScaler=residual_diffScaler,
    outputTimesteps=outputTimesteps
)

# print(fold3_pred.shape) # Shape [136, 8]
# print(fold3_actual.shape) # Shape [136, 8]
# print(fold3_price_before.shape) # Shape [136]

def visualize_fold_predictions(fold_pred, fold_actual, fold_price_before, fold_number):
    reshaped_preds = fold_price_before.reshape(-1, 1)
    for t in range(fold_pred.shape[1]):
        predictions = fold_pred[:, t]
        reshaped_preds = np.hstack([reshaped_preds, predictions.reshape(-1, 1)])

    first_actual = fold_price_before[:7]
    second_actual = fold_actual[:-7, 0]
    third_actual = fold_actual[-7:, :].T.flatten(order='C')
    combined_actual = np.concatenate([first_actual, second_actual, third_actual])

    plt.figure(figsize=(16, 8))
    plt.plot(
        combined_actual, 
        label='Actual Prices', 
        color='magenta', 
        linewidth=2, 
        alpha=0.9, 
        marker='o', 
        markersize=4
    )

    for row in range(len(reshaped_preds)):
        y_values = reshaped_preds[row]
        x_values = list(range(row, row + len(y_values) * interval, interval))[:len(y_values)]
        plt.plot(
            x_values, 
            y_values, 
            color=np.random.rand(3,), 
            linestyle='-', 
            linewidth=1.5, 
            alpha=0.8
        )

    plt.title(f'Oil Price Predict (Test) - Fold {fold_number}', fontsize=16)
    plt.xlabel('Time Period', fontsize=12)
    plt.ylabel('Price', fontsize=12)
    plt.grid(alpha=0.3)
    plt.legend(['Actual Prices', 'Predicted Sequences'], loc='upper right', fontsize=10)
    plt.tight_layout()
    plt.show()

# Calls the visualization function
visualize_fold_predictions(fold1_pred, fold1_actual, fold1_price_before, fold_number=1)
visualize_fold_predictions(fold2_pred, fold2_actual, fold2_price_before, fold_number=2)
visualize_fold_predictions(fold3_pred, fold3_actual, fold3_price_before, fold_number=3)
# -----------------------------------------------------------------------------------------------------------------------------------